In [2]:
# === Step 1: Import Your Toolkits ===
# What are we doing? We're gathering the tools we need from Python's giant toolbox.
# pandas: The ultimate tool for working with data tables (we call them DataFrames).
# os: This lets our script interact with the computer's operating system, like finding files in a folder.
# sqlalchemy: A powerful tool that lets Python speak the language of databases (SQL).
# logging: Our script's personal diary. It writes down everything it does and any errors it finds.
# time: A simple stopwatch to see how long our process takes.

import pandas as pd
import os
from sqlalchemy import create_engine
import logging
import time

# === Step 2: Set Up the Script's Diary (Logging) ===
# What are we doing? We are telling our script to create and save its diary in our project folder.
# Why? So we can easily find and read the log to check that everything worked correctly.
# This is a professional habit. It makes debugging (finding and fixing errors) much easier.

log_path = "brazil_project_ingestion.log" # The diary will be saved right here in our D:\Brazil_Vendor_Project folder.

logging.basicConfig(
    filename=log_path,           # The name of the diary file.
    level=logging.INFO,          # We want it to record all important events (INFO level).
    format="%(asctime)s - %(levelname)s - %(message)s", # The format for each diary entry: Time - Type of Entry - The Message.
    filemode='w'                 # 'w' means 'write'. Every time we run the script, it starts a fresh diary.
)

# === Step 3: Create the "Key" to Our Database ===
# What are we doing? We are defining where our database will live and what it's called.
# It will be a single file named 'brazil_ecommerce.db' in our main project folder.
# Why? All our organized data will be stored in this one file, making it easy to access later.

db_path = "brazil_ecommerce.db"
engine = create_engine('sqlite:///' + db_path) # This line creates the "engine" that will manage our database connection.
logging.info("Database engine created. It will be saved at: " + db_path)

# === Step 4: The Master Function to Run the Whole Process ===
# What are we doing? We are putting all our instructions into one big function called 'load_all_raw_data'.
# A function is like a recipe that you can call by name whenever you need it.
# Why? Organizing code into functions makes it clean, reusable, and easy to understand.

def load_all_raw_data():
    """Finds all CSVs in the 'data' folder and loads them into our database."""
    start_time = time.time() # Start the stopwatch.
    logging.info("--- Starting ETL Process ---")

    # This is the path to our data folder.
    data_folder = 'data'
    # os.listdir() looks inside this folder and gives us a list of all the file names.
    all_files_in_folder = os.listdir(data_folder)

    # This 'for loop' will go through our list of files, one by one.
    for filename in all_files_in_folder:
        # We only want to process files that end with '.csv'.
        if filename.endswith('.csv'):
            try: # 'try' is like saying, "Attempt to do this, but be ready for errors."
                # We build the full path to the file, e.g., 'data/olist_sellers_dataset.csv'
                full_file_path = os.path.join(data_folder, filename)

                # We create a clean table name by removing the '.csv' part from the filename.
                # 'olist_sellers_dataset.csv' becomes 'olist_sellers_dataset'.
                table_name = filename.replace('.csv', '').replace('olist_', '').replace('_dataset', '')
                if table_name == 'product_category_name_translation':
                    table_name = 'category_name_translation'

                logging.info("Processing file: " + filename + " -> into table: " + table_name)

                # Here, we use pandas to read the entire CSV file into memory.
                # These files are small enough that we don't need the 'chunking' method.
                df = pd.read_csv(full_file_path)

                # This is the magic 'Load' step.
                # df.to_sql() takes our pandas DataFrame (the table in memory) and saves it
                # as a table inside our SQLite database file.
                # name=table_name: The name of the table in the database.
                # con=engine: The database connection we want to use.
                # if_exists='replace': If a table with this name already exists, delete it and create a new one.
                # index=False: We don't need to save the pandas index number as a column.
                df.to_sql(name=table_name, con=engine, if_exists='replace', index=False)

                logging.info("Successfully loaded table: " + table_name)

            except Exception as e: # If any error happens inside the 'try' block...
                # ...we 'catch' the error here and record it in our diary instead of crashing the whole script.
                logging.error("Failed to process " + filename + ". Error: " + str(e))

    end_time = time.time() # Stop the stopwatch.
    total_time_seconds = end_time - start_time
    logging.info("--- ETL Process Complete. Total time taken: " + str(round(total_time_seconds, 2)) + " seconds. ---")

# === Step 5: The "Start Button" for our Script ===
# What is this? This is a standard Python safety check.
# It means: "Only run the 'load_all_raw_data' recipe if this script is being run directly."
# Why? It prevents the code from running automatically if we were to import it into another script.

if __name__ == '__main__':
    load_all_raw_data()
    print("ETL Process has finished. Check the log file 'brazil_project_ingestion.log' for details.")

ETL Process has finished. Check the log file 'brazil_project_ingestion.log' for details.
